# Can a Neural Network Predict the Mandelbrot Set?

This notebook accompanies the blog post on training vanilla and Fourier-feature neural networks on a continuous Mandelbrot target.

The notebook is fully self-contained. It will:

1. Generate a smooth escape-time target.
2. Build a mixed training set with both global and boundary-focused samples.
3. Train a vanilla coordinate MLP.
4. Train a Fourier-feature MLP.
5. Plot the true field and compare the two predictions.

It is deliberately written to be easy to edit, so you can play with `fourier_scale`, network width, number of training steps, and the sampling strategy.

In [ ]:
import math
from itertools import cycle

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
torch.manual_seed(0)
np.random.seed(0)

## 1. Continuous Mandelbrot Target

Instead of predicting a hard binary label, we use a continuous target based on the smooth escape time. Points inside the set stay at `1`, while points outside decay smoothly away from the boundary.

In [ ]:
def smooth_escape_target(coords: np.ndarray, max_iter: int = 120, tau: float = 5.0) -> np.ndarray:
    """
    Continuous Mandelbrot target:
    1 inside the set (up to max_iter), exp(-mu / tau) outside.
    """
    c = coords[:, 0] + 1j * coords[:, 1]
    z = np.zeros_like(c)
    active = np.ones(c.shape, dtype=bool)
    smooth = np.zeros(c.shape, dtype=np.float64)

    for n in range(max_iter):
        z[active] = z[active] * z[active] + c[active]
        escaped = active & (np.abs(z) > 2.0)

        if np.any(escaped):
            smooth[escaped] = (
                n + 1 - np.log(np.log(np.abs(z[escaped]))) / np.log(2.0)
            )

        active &= ~escaped
        if not np.any(active):
            break

    target = np.exp(-smooth / tau)
    target[active] = 1.0
    return target.astype(np.float32)


def make_grid(width: int = 700, height: int = 520) -> np.ndarray:
    xs = np.linspace(-2.0, 1.0, width, dtype=np.float32)
    ys = np.linspace(-1.25, 1.25, height, dtype=np.float32)
    xx, yy = np.meshgrid(xs, ys)
    return np.column_stack([xx.ravel(), yy.ravel()]).astype(np.float32)


grid_coords = make_grid()
grid_target = smooth_escape_target(grid_coords).reshape(520, 700)

plt.figure(figsize=(9, 6))
plt.imshow(grid_target, extent=[-2, 1, -1.25, 1.25], origin="lower", cmap="magma")
plt.colorbar(label="target value")
plt.title("Continuous Mandelbrot target")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

## 2. Training Data

A purely uniform sample over the full window tends to overrepresent boring points far from the boundary. To compensate, we mix a global sample with a few boundary-heavy boxes.

In [ ]:
def sample_training_coordinates(n_global: int, n_focus: int, seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)

    global_x = rng.uniform(-2.0, 1.0, n_global)
    global_y = rng.uniform(-1.25, 1.25, n_global)

    boxes = [
        (-0.95, -0.55,  0.00, 0.35),
        (-1.80, -1.20, -0.10, 0.10),
        ( 0.20,  0.45,  0.45, 0.75),
    ]

    per_box = n_focus // len(boxes)
    focus_chunks = []

    for x0, x1, y0, y1 in boxes:
        xs = rng.uniform(x0, x1, per_box)
        ys = rng.uniform(y0, y1, per_box)
        focus_chunks.append(np.column_stack([xs, ys]))

    coords = np.vstack([
        np.column_stack([global_x, global_y]),
        *focus_chunks,
    ])

    rng.shuffle(coords)
    return coords.astype(np.float32)


def make_dataset(
    n_global: int = 40000,
    n_focus: int = 12000,
    max_iter: int = 120,
    tau: float = 5.0,
    seed: int = 0,
) -> TensorDataset:
    coords = sample_training_coordinates(n_global, n_focus, seed=seed)
    target = smooth_escape_target(coords, max_iter=max_iter, tau=tau)
    x = torch.from_numpy(coords)
    y = torch.from_numpy(target[:, None])
    return TensorDataset(x, y)


dataset = make_dataset()
print(f"Dataset size: {len(dataset):,}")

## 3. Models

We train the same base MLP twice. The only change is the input encoding:

- `use_fourier=False`: raw coordinates `(x, y)`
- `use_fourier=True`: random Fourier features before the MLP

In [ ]:
class FourierFeatures(nn.Module):
    def __init__(self, input_dim: int = 2, mapping_size: int = 128, scale: float = 8.0) -> None:
        super().__init__()
        B = torch.randn(input_dim, mapping_size) * scale
        self.register_buffer("B", B)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_proj = 2.0 * math.pi * (x @ self.B)
        return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)


class FractalNet(nn.Module):
    def __init__(
        self,
        use_fourier: bool = True,
        mapping_size: int = 128,
        fourier_scale: float = 8.0,
        hidden_dim: int = 128,
        depth: int = 3,
    ) -> None:
        super().__init__()

        if use_fourier:
            self.mapping = FourierFeatures(2, mapping_size, fourier_scale)
            in_dim = 2 * mapping_size
        else:
            self.mapping = nn.Identity()
            in_dim = 2

        layers = [nn.Linear(in_dim, hidden_dim), nn.GELU()]
        for _ in range(depth - 1):
            layers.extend([nn.Linear(hidden_dim, hidden_dim), nn.GELU()])
        layers.extend([nn.Linear(hidden_dim, 1), nn.Sigmoid()])
        self.mlp = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.mlp(self.mapping(x))

## 4. Training and Evaluation Utilities

In [ ]:
def train_model(
    model: nn.Module,
    dataset: TensorDataset,
    steps: int = 1500,
    batch_size: int = 2048,
    lr: float = 2e-4,
    device: str = "cpu",
) -> list[float]:
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    iterator = cycle(loader)

    model.to(device)
    optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-6)
    loss_fn = nn.MSELoss()
    history = []

    for step in range(steps):
        x, y = next(iterator)
        x = x.to(device)
        y = y.to(device)

        optimiser.zero_grad(set_to_none=True)
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimiser.step()

        history.append(loss.item())
        if (step + 1) % 250 == 0:
            print(f"step {step + 1:4d} | loss = {loss.item():.6f}")

    return history


@torch.no_grad()
def evaluate_grid(model: nn.Module, width: int = 700, height: int = 520, device: str = "cpu") -> np.ndarray:
    xs = torch.linspace(-2.0, 1.0, width)
    ys = torch.linspace(-1.25, 1.25, height)
    yy, xx = torch.meshgrid(ys, xs, indexing="ij")
    coords = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=-1).to(device)
    pred = model(coords).reshape(height, width)
    return pred.cpu().numpy()

## 5. Run the Comparison

These defaults are chosen to run comfortably in Colab. If you have a GPU, try increasing `steps`, `mapping_size`, or `hidden_dim`.

In [ ]:
config = {
    "n_global": 40000,
    "n_focus": 12000,
    "max_iter": 120,
    "tau": 5.0,
    "steps": 1500,
    "batch_size": 2048,
    "lr": 2e-4,
    "mapping_size": 128,
    "fourier_scale": 8.0,
    "hidden_dim": 128,
    "depth": 3,
}

dataset = make_dataset(
    n_global=config["n_global"],
    n_focus=config["n_focus"],
    max_iter=config["max_iter"],
    tau=config["tau"],
)

vanilla = FractalNet(
    use_fourier=False,
    hidden_dim=config["hidden_dim"],
    depth=config["depth"],
)
fourier = FractalNet(
    use_fourier=True,
    mapping_size=config["mapping_size"],
    fourier_scale=config["fourier_scale"],
    hidden_dim=config["hidden_dim"],
    depth=config["depth"],
)

print("Training vanilla MLP...")
vanilla_history = train_model(
    vanilla,
    dataset,
    steps=config["steps"],
    batch_size=config["batch_size"],
    lr=config["lr"],
    device=device,
)

print("\nTraining Fourier-feature MLP...")
fourier_history = train_model(
    fourier,
    dataset,
    steps=config["steps"],
    batch_size=config["batch_size"],
    lr=config["lr"],
    device=device,
)

vanilla_pred = evaluate_grid(vanilla, device=device)
fourier_pred = evaluate_grid(fourier, device=device)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].imshow(grid_target, extent=[-2, 1, -1.25, 1.25], origin="lower", cmap="magma")
axes[0, 0].set_title("True target")

axes[0, 1].imshow(vanilla_pred, extent=[-2, 1, -1.25, 1.25], origin="lower", cmap="magma")
axes[0, 1].set_title("Vanilla MLP prediction")

axes[1, 0].imshow(fourier_pred, extent=[-2, 1, -1.25, 1.25], origin="lower", cmap="magma")
axes[1, 0].set_title("Fourier-feature prediction")

axes[1, 1].plot(vanilla_history, label="vanilla", lw=2)
axes[1, 1].plot(fourier_history, label="fourier", lw=2)
axes[1, 1].set_title("Training loss")
axes[1, 1].set_xlabel("step")
axes[1, 1].set_ylabel("MSE loss")
axes[1, 1].legend()

for ax in axes[:,:3].ravel():
    ax.set_xlabel("x")
    ax.set_ylabel("y")

plt.tight_layout()
plt.show()

## 6. Things to Tinker With

A few useful experiments:

- Increase `steps` and see how much more detail each model recovers.
- Change `fourier_scale` to something like `2.0`, `8.0`, `20.0` and compare.
- Reduce `n_focus` to zero and watch what happens when the network mostly sees the uninteresting background.
- Increase `mapping_size` if you want a richer Fourier basis.

If you want to zoom in on one particular minibrot region, you can also change the evaluation window and retrain on a more local dataset.